In [1]:
import os

# ── MUST set env vars BEFORE any app import ──
os.environ["REDIS_URL"] = "redis://localhost:6379/0"
os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "changeme"
os.environ["CHROMA_HOST"] = "localhost"
os.environ["CHROMA_PORT"] = "8000"
os.environ["SEARXNG_URL"] = "http://localhost:8080"
os.environ["OBSCURA_CDP_URL"] = "http://localhost:9222"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
client.__enter__()

/home/prithwijit/programming/python/imp_projects/micro_services_api/ai-infra-stack/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [2]:
# test_health()

#### ***health***

In [3]:
# _ctx = TestClient(app)
# client = _ctx.__enter__()

In [4]:
h = client.get("/health")
h.status_code, h.text, h.json()

(200, '{"status":"ok"}', {'status': 'ok'})

#### ***test_search***

In [5]:
s = client.post("/search", json={"query": "what is the hotel price of sikkim in makemytrip?", "max_results": 3})

In [7]:
s.json()

{'query': 'what is the hotel price of sikkim in makemytrip?',
 'number_of_results': 3,
 'results': [{'title': 'Hotels in Sikkim Get Best Deal Upto 50% OFF on Sikkim ... - MakeMyTrip',
   'url': 'https://www.makemytrip.com/hotels/best-hotels-in-sikkim.html',
   'content': 'Mountain whisper home stay Yuksom. Building, Yuksom Colony Near Knp Checkpost West Sikkim, Yuksom Colony, Yuksom · NORBUKHANG KANCHENSUNRISE HOMESTAY, Yuksom\xa0...',
   'engine': 'duckduckgo'},
  {'title': 'Sikkim Tour Packages Starts at ₹19225 Book Now! - MakeMyTrip',
   'url': 'https://www.makemytrip.com/holidays-india/sikkim-travel-packages.html',
   'content': 'A typical Sikkim holiday package can range from ₹15,000 to ₹50,000 per person, depending on the duration, type of accommodation, and inclusions like flights,\xa0...',
   'engine': 'duckduckgo'},
  {'title': 'Gangtok Hotels, resorts, homestays and more | MakeMytrip.com',
   'url': 'https://www.makemytrip.global/hotels-international/en-us/india/gangtok-hotel

#### ***crawl***

In [6]:
c = client.post("/crawl", json = {"url": "https://www.makemytrip.com/hotels/jaipur-hotels.html?msockid=3a19bc49b5fc625f2656abecb40c63bc"})
c.json()

static crawl timed out or failed for https://www.makemytrip.com/hotels/jaipur-hotels.html?msockid=3a19bc49b5fc625f2656abecb40c63bc: 
StealthyFetcher failed for https://www.makemytrip.com/hotels/jaipur-hotels.html?msockid=3a19bc49b5fc625f2656abecb40c63bc: BrowserType.launch_persistent_context: Executable doesn't exist at /home/prithwijit/.cache/ms-playwright/chromium-1228/chrome-linux64/chrome
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     patchright install                                     ║
║                                                            ║
║ <3 Patchright Team                                         ║
╚════════════════════════════════════════════════════════════╝


{'url': 'https://www.makemytrip.com/hotels/jaipur-hotels.html?msockid=3a19bc49b5fc625f2656abecb40c63bc',
 'markdown': '',
 'html': None,
 'title': None,
 'status_code': 200}

In [7]:
r = client.post("/crawl", json={
        "url": "https://www.makemytrip.com/hotels/jaipur-hotels.html",
        "timeout_ms": 60000,
    })
r.text

static crawl timed out or failed for https://www.makemytrip.com/hotels/jaipur-hotels.html: 
StealthyFetcher failed for https://www.makemytrip.com/hotels/jaipur-hotels.html: BrowserType.launch_persistent_context: Executable doesn't exist at /home/prithwijit/.cache/ms-playwright/chromium-1228/chrome-linux64/chrome
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     patchright install                                     ║
║                                                            ║
║ <3 Patchright Team                                         ║
╚════════════════════════════════════════════════════════════╝


'{"url":"https://www.makemytrip.com/hotels/jaipur-hotels.html","markdown":"","html":null,"title":null,"status_code":200}'

In [8]:
# Crawl engine selection is fixed by the service fallback chain

In [9]:
c = client.post("/crawl", json = {"url": "https://www.makemytrip.com/hotels/best-hotels-in-sikkim.html"})
c.json()

static crawl timed out or failed for https://www.makemytrip.com/hotels/best-hotels-in-sikkim.html: 
StealthyFetcher failed for https://www.makemytrip.com/hotels/best-hotels-in-sikkim.html: BrowserType.launch_persistent_context: Executable doesn't exist at /home/prithwijit/.cache/ms-playwright/chromium-1228/chrome-linux64/chrome
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     patchright install                                     ║
║                                                            ║
║ <3 Patchright Team                                         ║
╚════════════════════════════════════════════════════════════╝


{'url': 'https://www.makemytrip.com/hotels/best-hotels-in-sikkim.html',
 'markdown': '',
 'html': None,
 'title': None,
 'status_code': 200}

In [10]:
c = client.post("/crawl", json = {"url": "https://example.com"})
c.json()

{'url': 'https://example.com/',
 'markdown': '<doc fingerprint="bcbae6b725d8d3f0">\n  <main>\n    <p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p>\n    <p>Learn more</p>\n  </main>\n  <comments/>\n</doc>',
 'html': None,
 'title': 'Example Domain',
 'status_code': 200}

#### ***cache***

In [11]:
ca_set = client.post("/cache/set", json={"key": "test:smoke:foot", "value": {"a": 2}, "ttl_seconds": 60})
ca_set.json()

{'key': 'test:smoke:foot', 'success': True}

In [12]:
ca_get = client.get("cache/get/test:smoke:foo")
ca_get.json()

{'key': 'test:smoke:foo', 'value': None, 'found': False}

In [13]:
## delete too simple

#### ***Vector***

In [14]:
os.environ["CHROMA_HOST"], os.environ["CHROMA_PORT"]

('localhost', '8000')

In [15]:
import chromadb
from chromadb.config import Settings

cl = chromadb.HttpClient(host=os.environ["CHROMA_HOST"], port=int(os.environ["CHROMA_PORT"]), settings=Settings(anonymized_telemetry=False))
cl.heartbeat()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


1785325929614105342

In [16]:
coll = "test_smoke_vec"
ch_po = client.post("/vector/upsert", json={
        "collection": coll,
        "records": [{"id": "1", "embedding": [0.1, 0.2], "document": "hello", "metadata": {"src": "test"}}],
    })

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [17]:
ch_po2 = client.post("/vector/search", json={
        "collection": coll, "query_embedding": [0.1, 0.2], "top_k": 5,
    })

In [18]:
ch_po2.json()

{'collection': 'test_smoke_vec',
 'matches': [{'id': '1',
   'score': 0.0,
   'document': 'hello',
   'metadata': {'src': 'test'}}]}

In [19]:
ch_po.status_code, ch_po.text

(200, '{"collection":"test_smoke_vec","upserted":1}')

#### ***test_graph***

In [20]:
label = "TestPerson"
g = client.post("/graph/query", json={"cypher": f"MATCH (n:{label}) DETACH DELETE n"})
g.text

'{"records":[],"count":0}'

#### **yt-dlp***

In [21]:
import yt_dlp
r = client.post("/youtube/info", json={"url": "https://www.youtube.com/watch?v=dQw4w9WgXcQ"})

In [22]:
r.text

'{"id":"dQw4w9WgXcQ","title":"Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)","duration":213,"uploader":"Rick Astley","view_count":1797742254,"thumbnail":"https://i.ytimg.com/vi/dQw4w9WgXcQ/maxresdefault.jpg","webpage_url":"https://www.youtube.com/watch?v=dQw4w9WgXcQ"}'

#### ***CLIP***

In [23]:
clip = client.post("/clip/text_embedding", json={"texts": ["hello world"]})
clip.text

/home/prithwijit/programming/python/imp_projects/micro_services_api/ai-infra-stack/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 45620.16it/s]


'{"model":"openai/clip-vit-base-patch32","dimensions":512,"embeddings":[[0.1020461916923523,0.12730856239795685,-0.16999100148677826,0.24520361423492432,-0.08504176884889603,-0.24274194240570068,0.1644097864627838,-1.478373408317566,0.2049737125635147,-0.014116354286670685,-0.4083241820335388,-0.0925239622592926,-0.23295487463474274,-0.09101560711860657,0.22803136706352234,-0.04481073468923569,0.37784382700920105,0.03151395171880722,-0.024176251143217087,-0.0905928760766983,0.25476568937301636,-0.24197453260421753,0.21334445476531982,0.16667816042900085,-0.4167823791503906,-0.3470451235771179,-0.1378737986087799,0.2722877264022827,-0.13879543542861938,0.26067477464675903,0.1629432588815689,-0.1243935227394104,-0.004743784666061401,-0.11912097781896591,0.02568022906780243,0.12757061421871185,-0.08209186792373657,-0.037103861570358276,0.15544331073760986,-0.16989390552043915,0.011077404022216797,-0.16502606868743896,-0.09525424242019653,-0.17598956823349,0.19897358119487762,0.13881748914

#### ***embed***

In [24]:
embd = client.post("/embed", json={"texts": ["hello world"]})
embd.text

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10660.53it/s]


'{"model":"BAAI/bge-small-en-v1.5","dimensions":384,"embeddings":[[0.015196082182228565,-0.022570626810193062,0.008547124452888966,-0.07417058944702148,0.0038364348001778126,0.0027135401032865047,-0.031267885118722916,0.04463399946689606,0.04405517503619194,-0.007871159352362156,-0.02520078606903553,-0.033366523683071136,0.014427945017814636,0.04653816297650337,0.008555104024708271,-0.016145750880241394,0.007405815180391073,-0.019012467935681343,-0.11472629010677338,-0.01815764605998993,0.12635935842990875,0.02970287762582302,0.025280997157096863,-0.03421787917613983,-0.040999636054039,0.006617303006350994,0.010270580649375916,0.022362282499670982,0.004436321556568146,-0.12730969488620758,-0.01614920049905777,-0.020380130037665367,0.04721209406852722,0.011579862795770168,0.06818711757659912,0.007298627868294716,-0.017852995544672012,0.04078216850757599,-0.010269462130963802,0.023757122457027435,0.010602869093418121,-0.02858438901603222,0.008159694261848927,-0.015180516988039017,0.03089

#### ***browse***

In [25]:
browse = client.post("/browse", json={"url": "https://example.com", "action": "content"})
browse.text

'{"detail":"BrowserType.launch: Executable doesn\'t exist at /home/prithwijit/.cache/ms-playwright/chromium_headless_shell-1228/chrome-headless-shell-linux64/chrome-headless-shell\\n╔════════════════════════════════════════════════════════════╗\\n║ Looks like Playwright was just installed or updated.       ║\\n║ Please run the following command to download new browsers: ║\\n║                                                            ║\\n║     playwright install                                     ║\\n║                                                            ║\\n║ <3 Playwright Team                                         ║\\n╚════════════════════════════════════════════════════════════╝"}'

In [26]:
browse.json()

{'detail': "BrowserType.launch: Executable doesn't exist at /home/prithwijit/.cache/ms-playwright/chromium_headless_shell-1228/chrome-headless-shell-linux64/chrome-headless-shell\n╔════════════════════════════════════════════════════════════╗\n║ Looks like Playwright was just installed or updated.       ║\n║ Please run the following command to download new browsers: ║\n║                                                            ║\n║     playwright install                                     ║\n║                                                            ║\n║ <3 Playwright Team                                         ║\n╚════════════════════════════════════════════════════════════╝"}

#### ***reranker***

In [27]:
reranker = client.post("/rerank", json={
        "query": "python programming",
        "documents": ["python is a language", "cookies are tasty"],
    })

reranker.text

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 9922.95it/s]


'{"model":"BAAI/bge-reranker-v2-m3","query":"python programming","results":[{"index":0,"document":"python is a language","score":0.9952011108398438},{"index":1,"document":"cookies are tasty","score":0.00001614524626347702}]}'

#### ***full test***

In [9]:
from tests.test_smoke import test_health, test_search, test_crawl, test_cache, test_vector, test_graph, test_youtube, test_reranker, test_browse, test_embed, test_clip, results, record

tests = [
    test_health,
    test_search,
    test_crawl,
    test_cache,
    test_vector,
    test_graph,
    test_youtube,
    test_reranker,
    test_browse,
    test_embed,
    test_clip,
]
failed = 0
for t in tests:
    try:
        t()
    except Exception as e:
        failed += 1
        record(t.__name__, False, f"{type(e).__name__}: {e}")

print("\n" + "=" * 60)
passed = sum(1 for _, ok, _ in results if ok)
print(f"RESULTS: {passed}/{len(results)} passed")
print("=" * 60)


[PASS] health: endpoint returns ok
[PASS] search: SearXNG wrapper


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[PASS] crawl: real crawler (trafilatura)
[PASS] cache: Redis wrapper (set/get/delete)
[PASS] vector: ChromaDB wrapper (upsert/search/delete)
[PASS] graph: Neo4j wrapper + injection guard


[PASS] youtube: yt-dlp info wrapper


/home/prithwijit/programming/python/imp_projects/micro_services_api/ai-infra-stack/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10034.89it/s]


[PASS] reranker: route wiring + ranking order
[PASS] browse: Obscura cloud browser


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12152.98it/s]


[PASS] embed: real embedding model


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 41662.50it/s]


[PASS] clip: real CLIP text embedding

RESULTS: 11/11 passed
